# mamba3 on CUDA — training and inference on-device

Runs the `mamba3` crate's full learning + inference stack on this machine's
NVIDIA GPU through CubeCL's CUDA backend:

1. the full test suite (`--features cuda`),
2. **learning** — `train_lm` trains a Mamba-3 language model end to end on
   the GPU and reports held-out next-token accuracy,
3. **inference** — `generate` decodes with the recurrent state cache and
   checks the cached path against the parallel path bit-for-bit-ish
   (max |difference| on the logits),
4. **LoRA fine-tuning** and the **vision** model, same mixer,
5. matmul tuner verification (`MAMBA3_TUNE_CHECK`) plus a training-throughput
   benchmark on a 31.7M-parameter model.

**Before running:** Colab menu → *Runtime → Change runtime type* → **GPU**
(T4 is enough). The Rust dependency build takes ~10–15 minutes on Colab's
CPU the first time; every kernel is also JIT-compiled on first use, and the
repository's `cubecl.toml` persists that cache under `target/` so repeat runs
are fast.

In [ ]:
%%bash
set -e
if ! command -v nvidia-smi >/dev/null; then
  echo 'No GPU runtime. Colab menu: Runtime -> Change runtime type -> GPU.' >&2
  exit 1
fi
nvidia-smi

In [ ]:
%%bash
set -e
if [ ! -x "$HOME/.cargo/bin/cargo" ]; then
  curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs \
    | sh -s -- -y --profile minimal --default-toolchain stable >/dev/null
fi
$HOME/.cargo/bin/rustc --version
$HOME/.cargo/bin/cargo --version

In [ ]:
%%bash
set -e
cd /content
rm -rf mamba-trainer
git clone --depth 1 https://github.com/BectorVoom/mamba-trainer.git
cd mamba-trainer
git log -1 --oneline

## Full test suite on CUDA

Seven suites, 75 tests: tensor kernels, autograd gradient checks, the chunked
SSM scan against a sequential reference, model shapes, and a short end-to-end
training run. This cell is the long one — it builds the whole crate first.

In [ ]:
%%bash
set -eo pipefail
export PATH=$HOME/.cargo/bin:$PATH
cd /content/mamba-trainer
cargo test --release --no-default-features --features cuda 2>&1 \
  | tee /content/test.log \
  | grep -E 'Running|test result|FAILED|panicked'

## Learning: train a Mamba-3 LM on the GPU

Trains a 2-layer model on a synthetic stride-sequence task. Expected (same
numbers as the CPU, wgpu, SPIR-V and ROCm backends — initialisation is seeded
on the host, so any drift is a kernel disagreeing): loss ~3.19 → ~0.03 in 250
steps, held-out next-token accuracy **97.7%**, and the greedy continuation of
`[0, 2, 4, 6]` stepping by 2.

In [ ]:
%%bash
set -e
export PATH=$HOME/.cargo/bin:$PATH
cd /content/mamba-trainer
cargo run --release --no-default-features --features cuda --example train_lm

## Inference: incremental decoding with the state cache

The property the architecture exists for: after the prompt, each new token
costs the same no matter how much context precedes it (fixed-size recurrent
state, no KV cache). The example first checks the cached path reproduces the
parallel path (max |difference| should be ~1e-7), then times decoding at
8-token vs 512-token context — the two per-token times should match.

In [ ]:
%%bash
set -e
export PATH=$HOME/.cargo/bin:$PATH
cd /content/mamba-trainer
cargo run --release --no-default-features --features cuda --example generate

## LoRA fine-tuning and the vision model

`finetune_lora` freezes the base model and trains low-rank adapters
(~14% of parameters; verifies frozen tensors did not move, and that the
adapter checkpoint is small). `vision` trains the bidirectional-scan image
classifier to 100% held-out accuracy on a synthetic task.

In [ ]:
%%bash
set -e
export PATH=$HOME/.cargo/bin:$PATH
cd /content/mamba-trainer
cargo run --release --no-default-features --features cuda --example finetune_lora
echo
cargo run --release --no-default-features --features cuda --example vision

## Matmul tuner verification + training throughput

`MAMBA3_TUNE_CHECK=1` makes the tuner verify every matmul candidate against
the simple kernel on every shape a real 31.7M-parameter training step issues,
with the model's real operands — the cell fails loudly if any candidate
computes a wrong product. `MAMBA3_TUNE_LOG=1` prints which kernel won each
shape on **this** GPU. The second run reports steady-state tokens/s.

In [ ]:
%%bash
set -e
export PATH=$HOME/.cargo/bin:$PATH
cd /content/mamba-trainer
cargo build --release --no-default-features --features cuda --example bench_train
MAMBA3_TUNE_CHECK=1 MAMBA3_TUNE_LOG=1 ITERS=10 \
  ./target/release/examples/bench_train
echo
echo '=== steady-state (tuned, no checking) ==='
ITERS=15 ./target/release/examples/bench_train 2>/dev/null \
  | grep -E 'backend|params|step|tokens'

## Reduced precision, and the matrix cores

`MAMBA3_MATMUL_PRECISION=bf16` (or `f16`) rounds every matmul's operands to that
type once per call and accumulates in `f32` — master weights, gradients and every
other op stay `f32`. That halves the bytes a product moves, which the README's
shape analysis identified as the ceiling for the memory-bound half of them, and
it is the storage type the matrix cores take.

Both types are here because the hardware differs: **every** tensor-core
generation does `f16`, while `bf16` fragments arrived with Ampere and RDNA3. A
Colab **T4 is Turing — `f16` only**, so on a T4 the matrix-core candidate appears
under `f16` and not under `bf16`. On an A100/L4 both work. `bf16` remains the
training recommendation (its `f32`-sized exponent range needs no loss scaling);
`f16` is what makes the fragment path testable on the free tier.

Four things are checked below, per mode:

1. `MAMBA3_TUNE_CHECK=1` — every candidate, including the matrix-core one, is
   verified against the simple kernel *reading the same rounded operands*, so
   the tolerance stays meaningful. This is the real gate on the CMMA kernel.
2. `MAMBA3_TUNE_LOG=1` — look for `Cmma(..)` in the winning plans. Which shapes
   it wins is the interesting result: a fragment pipeline needs rows and columns
   to fill it, so the scan's 64-row batched products may still prefer a register
   kernel while the big projections do not.
3. Steady-state tokens/s vs the f32 rows above.
4. `train_lm` convergence under the mode (~97.7% held-out).


In [ ]:
%%bash
set -e
export PATH=$HOME/.cargo/bin:$PATH
cd /content/mamba-trainer

for MODE in bf16 f16; do
  echo "################ $MODE ################"
  export MAMBA3_MATMUL_PRECISION=$MODE
  echo '--- tuner check + winning plans (look for Cmma) ---'
  MAMBA3_TUNE_CHECK=1 MAMBA3_TUNE_LOG=1 ITERS=10 \
    ./target/release/examples/bench_train 2>&1 | grep -E 'tune|precision|step|tokens' | head -40
  echo
  echo "--- steady state, $MODE ---"
  ITERS=15 ./target/release/examples/bench_train 2>/dev/null \
    | grep -E 'backend|precision|params|step|tokens'
  echo
  echo "--- convergence gate, $MODE ---"
  cargo run --release --no-default-features --features cuda --example train_lm 2>/dev/null \
    | tail -4
  echo
done


## Reading the results

* **Tests** — all eight suites end `ok`, 84 tests total, 0 failed. The
  `mixed_precision` suite is the one that only fully runs here: its matrix-core
  test skips itself on a machine without tensor cores, which on a CPU is always.
* **train_lm** — 97.7% held-out accuracy and the stride-2 continuation, the
  same numbers every other backend produces.
* **generate** — max |difference| ~1e-7 between the parallel and cached
  paths, and per-token decode time flat from 8 to 512 tokens of context.
* **Tune check** — no aborts; the `tune ... ->` lines show the winning plan
  per shape on this GPU (winners may differ from the AMD iGPU the crate was
  developed on — that is the point of the tuner). If a shape looks slow,
  rerun with `MAMBA3_TUNE_LOG=2` to see every candidate's GFLOP/s.
* **Reduced precision** — the tune check passes under both modes, `train_lm`
  still reaches ~97.7%, and the steady-state rows show what halving operand
  traffic buys. Note which shapes chose `Cmma(..)`: that, and the GFLOP/s it
  reached under `MAMBA3_TUNE_LOG=2`, is the answer to whether the matrix-core
  path earns its place.

**The CMMA kernel has never run on hardware before this notebook does it.** It
is capability-gated, so it cannot affect a CPU or wgpu run, and
`MAMBA3_TUNE_CHECK` rejects a wrong product before the tuner is allowed to
prefer it — but the first execution of `matmul_cmma_kernel` anywhere is the
cell above. Treat a failure there as new-code triage, not as a regression.

If all of the above hold, Mamba-3 learning and inference are validated
on-device on CUDA.
